# EDPs com diferenças finitas

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/15-edp-diferencas-finitas.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. classificar uma EDP de segunda ordem como elíptica, parabólica ou hiperbólica, e saber que condições cada tipo pede;
> 2. discretizar as equações de Laplace e de Poisson com o estêncil de 5 pontos e resolvê-las por métodos diretos esparsos e iterativos (Jacobi, Gauss-Seidel, SOR);
> 3. discretizar a equação do calor com os esquemas explícito, implícito e de Crank-Nicolson, e deduzir as condições de estabilidade pela análise de von Neumann;
> 4. discretizar a advecção com os esquemas *upwind* e de Lax-Wendroff, explicar a condição CFL e reconhecer a difusão e a dispersão numéricas pela equação modificada;
> 5. explicar por que equações não lineares com choques, como a de Burgers, exigem a forma conservativa.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.optimize import brentq
from cnlivro.edp import adveccao_difusao, calor_theta, poisson_2d, burgers, passo_maximo

## 1. Problema de engenharia: fechar ou não a captação de água?

Às 6 h da manhã, um caminhão tomba numa ponte e derrama um produto químico num rio. A companhia de saneamento tem uma **captação de água** 18 km rio abaixo, e o limite para a água bruta é de 10 mg/L. Fechar a captação deixa a cidade sem água tratada; não fechar, se a concentração passar do limite, contamina a rede. A pergunta é: **qual será a concentração máxima na captação, e quando?**

Num rio, a concentração $c(x, t)$ média na seção transversal segue a **equação de advecção-difusão**

$$
\frac{\partial c}{\partial t} + u\,\frac{\partial c}{\partial x} = D\,\frac{\partial^2 c}{\partial x^2},
$$ (eq-adv-dif)

em que $u$ é a velocidade média da água e $D$ o coeficiente de **dispersão longitudinal**, que resume a mistura causada pela turbulência e pela diferença de velocidade entre o centro e as margens. O primeiro termo do lado esquerdo diz que a mancha **viaja** com a água; o do lado direito, que ela **se espalha**.

Neste trecho do rio, $u = 0{,}5$ m/s e $D = 5$ m²/s. A mancha, logo após o derramamento, tem a forma de uma gaussiana com pico de 50 mg/L e desvio-padrão de 150 m, centrada no km 2. Num trecho com velocidade e dispersão constantes, a equação tem solução exata, que usaremos como referência; num rio real, com seções, afluentes e velocidades que variam, só a solução numérica está disponível. O nosso trabalho é escolher um método que acerte este caso simples, para então confiar nele nos outros.

In [ ]:
u_rio, D_rio = 0.5, 5.0                 # m/s, m²/s
L_rio = 30_000.0                        # m (trecho modelado)
x_derr, x_capt = 2_000.0, 20_000.0      # m (derramamento e captação)
sigma0, pico0 = 150.0, 50.0             # m, mg/L
limite = 10.0                           # mg/L

def c_exata(x, t):
    # gaussiana que viaja com a água e se alarga: variância σ0² + 2 D t
    var = sigma0**2 + 2 * D_rio * t
    return pico0 * sigma0 / np.sqrt(var) * np.exp(-(x - x_derr - u_rio * t) ** 2 / (2 * var))

T_chegada = (x_capt - x_derr) / u_rio
t1 = brentq(lambda t: c_exata(x_capt, t) - limite, 0.9 * T_chegada, T_chegada)
t2 = brentq(lambda t: c_exata(x_capt, t) - limite, T_chegada, 1.1 * T_chegada)
print(f"a mancha chega à captação em {T_chegada / 3600:.1f} h, com pico de {c_exata(x_capt, T_chegada):.2f} mg/L")
print(f"acima do limite entre {t1 / 3600:.2f} h e {t2 / 3600:.2f} h após o derramamento ({(t2 - t1) / 60:.0f} min)")

In [ ]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1,
                    subplot_titles=["a mancha ao longo do rio", "a concentração na captação"])
xs = np.linspace(0, L_rio, 3001)
cores_t = [CORES["exata"], CORES["secundaria"], CORES["terciaria"], CORES["referencia"], CORES["aproximacao"]]
for cor, horas in zip(cores_t, [0, 2.5, 5, 7.5, 10]):
    fig.add_trace(go.Scatter(x=xs / 1000, y=c_exata(xs, horas * 3600), mode="lines", name=f"{numero(horas, 1)} h",
                             line=dict(color=cor, width=2.5)), 1, 1)
fig.add_vline(x=x_capt / 1000, line=dict(color=CORES["destaque"], dash="dash"), row=1, col=1)
fig.add_annotation(x=x_capt / 1000, y=45, text="captação", showarrow=False, xanchor="left", xshift=5,
                   font=dict(color=CORES["destaque"]), row=1, col=1)
th = np.linspace(8, 12, 400)
fig.add_trace(go.Scatter(x=th, y=c_exata(x_capt, th * 3600), mode="lines", showlegend=False,
                         line=dict(color=CORES["aproximacao"], width=3)), 1, 2)
fig.add_trace(go.Scatter(x=[8, 12], y=[limite, limite], mode="lines", name="limite, 10 mg/L",
                         line=dict(color=CORES["erro"], dash="dash")), 1, 2)
fig.update_xaxes(title="posição no rio (km)", row=1, col=1)
fig.update_yaxes(title="concentração (mg/L)", range=[0, 52], row=1, col=1)
fig.update_xaxes(title="tempo após o derramamento (h)", row=1, col=2)
fig.update_yaxes(title="concentração (mg/L)", range=[0, 14], row=1, col=2)
fig.update_layout(width=980, height=460, title="A mancha viaja, se espalha e passa da captação acima do limite",
                  legend=dict(orientation="h", y=-0.22), margin=dict(t=110))
mostrar(fig)

**O que notar:** em 10 horas, o pico cai de 50 para pouco mais de 12 mg/L, porque a mancha se alarga; a área sob cada curva (a massa derramada) não muda. Na captação, a concentração passa do limite por 26 minutos: a captação **tem de ser fechada**, e a janela é estreita. A seção 6 mostra como um código razoável chega à conclusão oposta.

## 2. A ideia em uma imagem

Uma EDP de evolução, discretizada por diferenças finitas, vira uma regra de atualização: o valor num ponto da malha, no próximo instante, é calculado a partir de alguns valores vizinhos no instante atual. Na advecção pura ($D = 0$), a informação viaja com velocidade $u$: o valor exato em $x_i$ no instante $t_{n+1}$ é o que estava em $x_i - u\,\Delta t$ no instante $t_n$. O esquema *upwind* ("contra o vento") calcula $c_i^{n+1}$ a partir de $c_{i-1}^n$ e $c_i^n$, os dois pontos rio acima. Se o ponto de partida $x_i - u\,\Delta t$ cair **fora** desse par, o esquema está tentando calcular algo a partir de informações que ele não usa. Arraste o slider para mudar o **número de Courant** $C = u\,\Delta t/\Delta x$, que mede quantas células a informação anda num passo.

In [ ]:
x_p = np.linspace(0, 1, 101)[:-1]
pulso = np.where(np.abs(x_p - 0.3) < 0.1, 1.0, 0.0)

def quadro_courant(Cn):
    fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.12,
                        subplot_titles=["o estêncil e a característica", "o pulso depois de uma volta completa"])
    # estêncil: dois níveis de tempo
    for nivel, cor in [(0, CORES["referencia"]), (1, CORES["referencia"])]:
        fig.add_trace(go.Scatter(x=[-2, -1, 0, 1], y=[nivel] * 4, mode="markers", showlegend=False,
                                 marker=dict(color="white", size=11, line=dict(color=cor, width=2))), 1, 1)
    fig.add_trace(go.Scatter(x=[-1, 0], y=[0, 0], mode="markers", showlegend=False,
                             marker=dict(color=CORES["aproximacao"], size=13)), 1, 1)
    fig.add_trace(go.Scatter(x=[0], y=[1], mode="markers", showlegend=False,
                             marker=dict(color=CORES["destaque"], size=13)), 1, 1)
    fig.add_trace(go.Scatter(x=[0, -Cn], y=[1, 0], mode="lines+markers", showlegend=False,
                             line=dict(color=CORES["erro"], width=3, dash="dash"),
                             marker=dict(symbol="x", size=[1, 14], color=CORES["erro"])), 1, 1)
    fig.add_annotation(x=-Cn, y=-0.18, text=f"x<sub>i</sub> − u Δt", showarrow=False,
                       font=dict(color=CORES["erro"]), row=1, col=1)
    fig.add_annotation(x=0, y=1.18, text="c<sub>i</sub><sup>n+1</sup>", showarrow=False, row=1, col=1)
    fig.add_annotation(x=-1.2, y=0.2, text="usados pelo upwind", showarrow=False,
                       font=dict(color=CORES["aproximacao"], size=12), row=1, col=1)
    r = adveccao_difusao(pulso, x_p, 1.0, 0.0, 1.0, dt=Cn / 100, periodico=True)
    fig.add_trace(go.Scatter(x=x_p, y=pulso, mode="lines", line=dict(color=CORES["exata"], dash="dash", shape="hv"),
                             name="exata"), 1, 2)
    fig.add_trace(go.Scatter(x=x_p, y=np.clip(r.u[-1], -2, 3), mode="lines", line=dict(color=CORES["aproximacao"], width=3),
                             name="upwind"), 1, 2)
    fig.update_xaxes(title="posição (em células, relativa a x<sub>i</sub>)", range=[-2.5, 1.5], row=1, col=1)
    fig.update_yaxes(title="nível de tempo", range=[-0.4, 1.4], tickvals=[0, 1], ticktext=["n", "n+1"], row=1, col=1)
    fig.update_xaxes(title="x", row=1, col=2)
    fig.update_yaxes(title="c", range=[-1.2, 2.2], row=1, col=2)
    amp = np.max(np.abs(r.u[-1]))
    estado = "dentro do estêncil: estável" if Cn <= 1 else f"fora do estêncil: instável (máximo de |c| = {amp:.0e})"
    fig.update_layout(width=980, height=430, showlegend=False, margin=dict(t=110),
                      title=f"C = {numero(Cn, 2)}: o pé da característica está {estado}")
    return fig

quadros = [(numero(v, 2), quadro_courant(v)) for v in [0.5, 0.9, 1.0, 1.1]]
mostrar(com_controle(quadros, rotulo="número de Courant C = ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com $C = 1$, o pé da característica cai exatamente em $x_{i-1}$ e o esquema copia o valor de lá: o pulso dá a volta sem nenhum erro. Com $C < 1$, o esquema faz uma média ponderada de dois valores, e cada média arredonda os cantos: o pulso chega borrado, mais quanto menor $C$. Com $C > 1$, o pé cai fora do estêncil e a solução explode. Essa é a **condição CFL** (de Courant, Friedrichs e Lewy, 1928): o domínio de dependência numérico precisa conter o físico. O borrado com $C < 1$ é a **difusão numérica**, a protagonista deste capítulo.

## 3. Formulação matemática

### 3.1 Três tipos de equação

Uma EDP linear de segunda ordem em duas variáveis,

$$
A\,u_{xx} + B\,u_{xy} + C\,u_{yy} + (\text{termos de ordem menor}) = 0,
$$

é classificada pelo sinal do discriminante $B^2 - 4AC$, como as cônicas:

| Tipo | $B^2 - 4AC$ | Exemplo | O que ela descreve | O que precisa |
| --- | --- | --- | --- | --- |
| elíptica | $< 0$ | Laplace $u_{xx} + u_{yy} = 0$, Poisson | regimes permanentes, equilíbrio | condições em **todo** o contorno |
| parabólica | $= 0$ | calor $u_t = \alpha u_{xx}$ | difusão, que suaviza | condição inicial + condições de contorno |
| hiperbólica | $> 0$ | onda $u_{tt} = a^2 u_{xx}$; advecção $u_t + a u_x = 0$ (1ª ordem) | propagação com velocidade finita | condição inicial + contorno onde a informação **entra** |

A classificação não é só nomenclatura: ela diz que métodos usar. Numa equação elíptica, cada ponto depende de todos os outros, e a discretização dá um grande **sistema linear** (capítulo 4). Nas parabólicas e hiperbólicas, marchamos no tempo, e a pergunta central é a **estabilidade** (capítulo 13). Nas hiperbólicas, a informação anda por **características**, e o esquema precisa respeitar a direção do vento. A advecção-difusão {eq}`eq-adv-dif` é parabólica, mas com $D$ pequeno se comporta quase como hiperbólica: é aí que mora a dificuldade.

### 3.2 Laplace e Poisson: o estêncil de 5 pontos

Para $-\nabla^2 u = f$ num retângulo, com malha uniforme $h$ nas duas direções, as diferenças centradas do capítulo 10 dão

$$
\frac{u_{i+1,j} - 2u_{i,j} + u_{i-1,j}}{h^2} + \frac{u_{i,j+1} - 2u_{i,j} + u_{i,j-1}}{h^2} = -f_{i,j}
\quad\Longleftrightarrow\quad
u_{i,j} = \frac{u_{i+1,j} + u_{i-1,j} + u_{i,j+1} + u_{i,j-1}}{4} + \frac{h^2}{4}f_{i,j},
$$ (eq-5pontos)

com erro de truncamento $O(h^2)$. Sem fonte, cada valor é a **média dos quatro vizinhos**: daí o princípio do máximo discreto (o máximo e o mínimo estão no contorno). Com $n$ subintervalos em cada direção, são $(n-1)^2$ incógnitas e um sistema esparso, com no máximo 5 coeficientes não nulos por linha. Há dois caminhos (capítulo 4):

- **direto esparso** (`scipy.sparse.linalg.spsolve`), que fatora a matriz aproveitando a esparsidade; ótimo em 2D até milhões de incógnitas;
- **iterativos**: **Jacobi** substitui todos os valores pela média {eq}`eq-5pontos` dos vizinhos antigos; **Gauss-Seidel** usa os novos assim que calculados; **SOR** (sobrerrelaxação sucessiva) exagera a correção de Gauss-Seidel por um fator $\omega \in (1, 2)$:

$$
u_{i,j} \leftarrow (1 - \omega)\,u_{i,j} + \omega\,\bar u_{i,j}, \qquad \omega_{\text{ótimo}} = \frac{2}{1 + \sin(\pi/n)} \ \text{(quadrado)}.
$$

Para o quadrado, o fator de convergência de Jacobi é $\rho_J = \cos(\pi/n) \approx 1 - \pi^2/(2n^2)$: o número de iterações cresce como $n^2$. Gauss-Seidel tem $\rho_{GS} = \rho_J^2$ (metade das iterações) e o SOR ótimo, $\rho \approx 1 - 2\pi/n$: as iterações crescem só como $n$.

### 3.3 Calor: explícito, implícito e Crank-Nicolson

Para $u_t = \alpha u_{xx}$, com $u_i^n \approx u(x_i, t_n)$ e $r = \alpha\,\Delta t/\Delta x^2$:

$$
\begin{aligned}
\text{FTCS (explícito):}\quad & u_i^{n+1} = u_i^n + r\,(u_{i+1}^n - 2u_i^n + u_{i-1}^n),\\
\text{BTCS (implícito):}\quad & u_i^{n+1} - r\,(u_{i+1}^{n+1} - 2u_i^{n+1} + u_{i-1}^{n+1}) = u_i^n,\\
\text{Crank-Nicolson:}\quad & u_i^{n+1} - \tfrac r2\,\delta^2 u_i^{n+1} = u_i^n + \tfrac r2\,\delta^2 u_i^n,
\end{aligned}
$$

com $\delta^2 u_i = u_{i+1} - 2u_i + u_{i-1}$. Os implícitos pedem a solução de um sistema **tridiagonal** por passo, de custo $O(n)$. FTCS (*forward time, centered space*) e BTCS têm ordem 1 no tempo e 2 no espaço; Crank-Nicolson, ordem 2 nos dois.

**Análise de von Neumann.** Como no capítulo 13, testamos o esquema num modo de Fourier, $u_j^n = g^n e^{\,\mathrm{i} j\theta}$, com $\theta = k\,\Delta x \in [0, \pi]$. Substituindo no FTCS, $e^{\mathrm{i}\theta} - 2 + e^{-\mathrm{i}\theta} = -4\sin^2(\theta/2)$, e o **fator de amplificação** é

$$
g_{\text{FTCS}} = 1 - 4r\sin^2\frac\theta2, \qquad g_{\text{BTCS}} = \frac{1}{1 + 4r\sin^2\frac\theta2}, \qquad
g_{\text{CN}} = \frac{1 - 2r\sin^2\frac\theta2}{1 + 2r\sin^2\frac\theta2}.
$$

O esquema é estável se $|g| \le 1$ para todo $\theta$. O FTCS exige $|1 - 4r| \le 1$, isto é,

$$
r = \frac{\alpha\,\Delta t}{\Delta x^2} \le \frac12 .
$$ (eq-ftcs)

O passo de tempo cai com o **quadrado** do espaçamento: dividir $\Delta x$ por 10 multiplica o número de passos por 100. BTCS e Crank-Nicolson são estáveis para qualquer $r$. Mas, para $r$ grande, $g_{\text{CN}} \to -1$ nos modos de alta frequência: eles não são amortecidos, apenas trocam de sinal a cada passo. É a mesma diferença entre A-estabilidade e L-estabilidade do capítulo 13, e a seção 7.3 mostra o que ela causa.

### 3.4 Advecção: centrado, upwind, Lax-Wendroff e a condição CFL

Para $c_t + u\,c_x = 0$ com $u > 0$ e $C = u\,\Delta t/\Delta x$:

$$
\begin{aligned}
\text{centrado (FTCS):}\quad & c_i^{n+1} = c_i^n - \tfrac C2\,(c_{i+1}^n - c_{i-1}^n), & g &= 1 - \mathrm{i}\,C\sin\theta,\\
\text{upwind:}\quad & c_i^{n+1} = c_i^n - C\,(c_i^n - c_{i-1}^n), & g &= 1 - C\,(1 - e^{-\mathrm{i}\theta}),\\
\text{Lax-Wendroff:}\quad & c_i^{n+1} = c_i^n - \tfrac C2\,(c_{i+1}^n - c_{i-1}^n) + \tfrac{C^2}{2}\,\delta^2 c_i^n, & g &= 1 - \mathrm{i}\,C\sin\theta - 2C^2\sin^2\tfrac\theta2.
\end{aligned}
$$

- O **centrado** tem $|g|^2 = 1 + C^2\sin^2\theta > 1$: é **instável para todo** $\Delta t$, por menor que seja. A combinação natural, "Euler no tempo e centrado no espaço", que funciona para o calor, não funciona para a advecção.
- O **upwind** usa o ponto rio acima. $|g| \le 1$ se e só se $0 \le C \le 1$: a **condição CFL**. Tem ordem 1.
- **Lax-Wendroff** vem de Taylor no tempo, $c^{n+1} = c + \Delta t\,c_t + \tfrac{\Delta t^2}{2}c_{tt}$, trocando as derivadas no tempo pelas no espaço usando a própria equação ($c_t = -u c_x$, $c_{tt} = u^2 c_{xx}$). Tem ordem 2 e é estável para $C \le 1$.

Com difusão, somamos $r\,\delta^2 c_i^n$ (com $r = D\,\Delta t/\Delta x^2$) a cada esquema. Em Lax-Wendroff, para manter a ordem 2, a expansão de Taylor usa $c_{tt} = u^2 c_{xx} - 2uD\,c_{xxx} + D^2 c_{xxxx}$, o que acrescenta dois termos pequenos, $-C\,r\,\delta^3 c_i^n + \tfrac{r^2}{2}\,\delta^4 c_i^n$ (com as diferenças centradas de terceira e quarta ordem); a função `adveccao_difusao` do pacote os inclui. A mesma análise dá as condições $C + 2r \le 1$ para o upwind e $C^2 + 2r \le 1$ para Lax-Wendroff; o centrado passa a ser estável se $C^2 \le 2r \le 1$ (a função `passo_maximo` do pacote calcula o maior $\Delta t$ de cada um).

### 3.5 A equação modificada: o que o esquema realmente resolve

Expandindo cada termo do upwind por Taylor em torno de $(x_i, t_n)$ e usando a própria equação para trocar derivadas no tempo, descobre-se que o upwind resolve, com erro $O(\Delta x^2)$, não a advecção pura, mas

$$
c_t + u\,c_x = D_{\text{num}}\,c_{xx}, \qquad D_{\text{num}} = \frac{u\,\Delta x}{2}\,(1 - C).
$$ (eq-dnum)

Essa é a **equação modificada** do esquema: o erro de primeira ordem se comporta como uma **difusão artificial**, que é zero com $C = 1$ e máxima com $C$ pequeno, exatamente o que a figura da seção 2 mostrou. Para o rio, com $\Delta x = 200$ m e $C = 0{,}5$, $D_{\text{num}} = 25$ m²/s, cinco vezes a dispersão física.

Em Lax-Wendroff, o termo $c_{xx}$ desaparece da equação modificada, e o erro principal é **dispersivo**, proporcional a $c_{xxx}$: ondas curtas andam com velocidade errada, e aparecem oscilações atrás das frentes íngremes. O **teorema de Godunov** (1959) diz que isso é inevitável: nenhum esquema linear de ordem maior que 1 preserva a monotonicidade (não cria máximos ou mínimos novos). Esquemas modernos contornam o teorema com **limitadores** não lineares (TVD, WENO), que usam ordem 2 onde a solução é suave e ordem 1 nas frentes.

### 3.6 Leis de conservação e a equação de Burgers

Muitas EDPs de engenharia são **leis de conservação**, $q_t + F(q)_x = 0$: a quantidade $q$ (massa, carros numa estrada, quantidade de movimento) só muda pelo fluxo $F$ através das fronteiras. O exemplo mais simples não linear é a **equação de Burgers**,

$$
u_t + \Big(\frac{u^2}{2}\Big)_x = 0 \quad\Longleftrightarrow\quad u_t + u\,u_x = 0 \ \ (\text{se } u \text{ for suave}),
$$

em que a velocidade de propagação é o próprio $u$: as partes mais rápidas alcançam as lentas e formam um **choque**, uma descontinuidade. No choque, a forma $u\,u_x$ não tem sentido, mas a conservação continua valendo e fixa a velocidade do choque (condição de Rankine-Hugoniot): $s = (u_E + u_D)/2$, a média dos estados à esquerda e à direita. Um esquema numérico só acerta essa velocidade se for escrito na **forma conservativa**, como diferença de fluxos entre faces de células,

$$
u_i^{n+1} = u_i^n - \frac{\Delta t}{\Delta x}\big(F_{i+1/2} - F_{i-1/2}\big),
$$

que é a ideia dos **volumes finitos** (teorema de Lax-Wendroff, 1960). A seção 7.4 mostra o que acontece sem ela.

## 4. À mão

**Laplace numa placa.** Uma placa quadrada tem a borda de cima a 100 °C e as outras três a 0 °C. Com $h = 1/3$, há quatro nós interiores; pela simetria, os dois de cima têm o mesmo valor $a$ e os dois de baixo, $b$. A média dos vizinhos {eq}`eq-5pontos` dá

$$
4a = 100 + a + b + 0, \qquad 4b = a + b + 0 + 0 \quad\Longrightarrow\quad 3a - b = 100, \quad a = 3b,
$$

logo $b = 12{,}5$ °C e $a = 37{,}5$ °C. Duas iterações de Jacobi a partir de zero: $(a, b) = (25;\ 0)$ e depois $(31{,}25;\ 6{,}25)$. Ainda longe: Jacobi é lento.

**FTCS e o limite $r \le 1/2$.** Uma barra com 5 nós tem $u = (0, 0, 100, 0, 0)$, com as pontas fixas em 0. Um passo de FTCS no nó central:

- com $r = 0{,}25$: $u_2 = 100 + 0{,}25\,(0 - 200 + 0) = 50$, e os vizinhos ficam com $0{,}25 \cdot 100 = 25$. O calor se espalha;
- com $r = 1$: $u_2 = 100 + (0 - 200 + 0) = -100$. Um ponto que estava mais quente que os vizinhos fica mais **frio** que eles: o esquema viola o princípio do máximo, e a oscilação cresce a cada passo.

**Upwind e a difusão numérica.** Uma frente $c = (1, 1, 0, 0)$ avança com $C = 0{,}5$: $c_2^{1} = 0 - 0{,}5\,(0 - 1) = 0{,}5$, e o resto não muda: $(1, 1, 0{,}5, 0)$. No passo seguinte, $(1, 1, 0{,}75, 0{,}25)$. A frente, que deveria continuar um degrau, vira uma rampa. Com $C = 1$: $(1, 1, 1, 0)$, o degrau exato deslocado de uma célula.

In [ ]:
# conferindo o Laplace à mão com o pacote
r = poisson_2d(0, lambda X, Y: np.where(np.isclose(Y, 1), 100.0, 0.0), 1, 1, 3, 3, "direto")
print("nós interiores (linha de baixo e de cima):", np.round(r.u[1:3, 1:3].T, 4).tolist())
for k in [1, 2]:
    rj = poisson_2d(0, lambda X, Y: np.where(np.isclose(Y, 1), 100.0, 0.0), 1, 1, 3, 3, "jacobi", max_iter=k)
    print(f"Jacobi, iteração {k}:", np.round(rj.u[1:3, 1:3].T, 4).tolist())

## 5. Explore

### 5.1 O limite do FTCS

Uma barra com 20 subintervalos tem o terço central inicialmente a 1 e o resto a 0, com as pontas em 0. Integramos até $t = 0{,}1$ com FTCS. Mude $r$ e observe o que acontece logo acima do limite $r = 1/2$.

In [ ]:
x_b = np.linspace(0, 1, 21)
u_b0 = np.where((x_b > 0.3) & (x_b < 0.7), 1.0, 0.0)
x_fino = np.linspace(0, 1, 401)
ref_b = calor_theta(np.where((x_fino > 0.3) & (x_fino < 0.7), 1.0, 0.0), x_fino, 1.0, 0.1, 1e-4, 0.5).u[-1]

def quadro_ftcs(r):
    res = calor_theta(u_b0, x_b, 1.0, 0.1, r / 400, 0.0)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_fino, y=ref_b, mode="lines", name="referência (malha fina)",
                             line=dict(color=CORES["exata"], dash="dash")))
    fig.add_trace(go.Scatter(x=x_b, y=np.clip(res.u[-1], -3, 3), mode="lines+markers", name="FTCS",
                             line=dict(color=CORES["aproximacao"], width=3), marker=dict(size=7)))
    fig.update_xaxes(title="posição x")
    fig.update_yaxes(title="temperatura u", range=[-1.5, 1.5])
    fig.update_layout(width=780, height=420, legend=dict(x=0.02, y=0.98),
                      title=f"r = {numero(r, 2)}, {res.passos} passos: máximo de |u| = {numero(np.abs(res.u[-1]).max(), 3)}")
    return fig

quadros = [(numero(r, 2), quadro_ftcs(r)) for r in [0.4, 0.5, 0.52, 0.6]]
mostrar(com_controle(quadros, rotulo="r = ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** até $r = 1/2$, a solução é suave e coincide com a referência. Com $r = 0{,}52$, apenas 4 % acima do limite, o modo de frequência mais alta (que alterna de sinal de um nó para o vizinho) é amplificado por $|1 - 4r| = 1{,}08$ a cada passo, e em 77 passos ele já domina. Com $r = 0{,}6$ o máximo passa de $10^7$. Não há meio-termo: o esquema é estável ou explode, e a explosão começa sempre pelas oscilações de um nó para o outro, a assinatura da instabilidade.

### 5.2 Laplace: quantas iterações?

A mesma placa da seção 4, agora com malhas finas. Compare como Jacobi e SOR se aproximam da solução direta, iteração por iteração, numa malha de 40 × 40.

In [ ]:
borda_quente = lambda X, Y: np.where(np.isclose(Y, 1), 100.0, 0.0)
n_p = 40
direta = poisson_2d(0, borda_quente, 1, 1, n_p, n_p, "direto")

def quadro_laplace(k):
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.06,
                        subplot_titles=[f"Jacobi, {k} iterações", f"SOR (ω ótimo), {k} iterações", "solução direta"])
    erros = []
    for col, U in enumerate([poisson_2d(0, borda_quente, 1, 1, n_p, n_p, "jacobi", max_iter=k).u,
                             poisson_2d(0, borda_quente, 1, 1, n_p, n_p, "sor", max_iter=k).u, direta.u], start=1):
        fig.add_trace(go.Heatmap(z=U.T, x=direta.x[0], y=direta.x[1], zmin=0, zmax=100, colorscale="Inferno",
                                 showscale=(col == 3), colorbar=dict(title="°C")), 1, col)
        erros.append(np.max(np.abs(U - direta.u)))
    for col in (1, 2, 3):
        fig.update_xaxes(title="x", row=1, col=col, scaleanchor=f"y{'' if col == 1 else col}", constrain="domain")
        fig.update_yaxes(showticklabels=(col == 1), row=1, col=col)
    fig.update_layout(width=980, height=400, margin=dict(t=110),
                      title=f"Erro máximo: Jacobi {numero(erros[0], 3)} °C, SOR {numero(erros[1], 3)} °C")
    return fig

quadros = [(str(k), quadro_laplace(k)) for k in [10, 50, 200, 1000]]
mostrar(com_controle(quadros, rotulo="iterações: ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** o calor da borda quente "se difunde" para dentro da placa iteração a iteração, uma camada de nós por vez no Jacobi: é como resolver uma equação do calor até o regime permanente, e é por isso que é lento. O SOR chega à solução em umas 100 iterações. O custo cresce com a malha:

In [ ]:
print("  n    Jacobi  Gauss-Seidel   SOR   (iterações até variação < 1e-6)   erro de Jacobi ao parar")
for n in [10, 20, 40, 80]:
    ref = poisson_2d(0, borda_quente, 1, 1, n, n, "direto").u
    its = {m: poisson_2d(0, borda_quente, 1, 1, n, n, m, tol=1e-6) for m in ["jacobi", "gauss-seidel", "sor"]}
    print(f"{n:4d} {its['jacobi'].passos:8d} {its['gauss-seidel'].passos:10d} {its['sor'].passos:8d}"
          f"{np.max(np.abs(its['jacobi'].u - ref)):42.1e} °C")

Dobrar $n$ quase quadruplica as iterações de Jacobi e de Gauss-Seidel (que faz a metade), e só dobra as do SOR, como a seção 3.2 previu. Repare na última coluna: o critério de parada "a variação numa iteração ficou abaixo de $10^{-6}$" deixa um erro mil vezes maior em Jacobi na malha de 80. Quando o fator de convergência $\rho$ está perto de 1, o erro restante é cerca de $\text{variação}/(1 - \rho)$: **uma iteração que anda pouco não é uma iteração que chegou**. Em 2D, com até alguns milhões de incógnitas, a solução direta esparsa é em geral a escolha mais simples; os iterativos (em especial gradiente conjugado com pré-condicionador e *multigrid*) tomam a frente em 3D.

### 5.3 A difusão numérica, medida

Uma gaussiana estreita e um pulso quadrado dão uma volta completa num domínio periódico de 200 células, com advecção pura. Comparamos o upwind e Lax-Wendroff com a solução exata (que só se desloca), e com a previsão da equação modificada {eq}`eq-dnum`: a gaussiana alargada por uma difusão $D_{\text{num}}$.

In [ ]:
x_m = np.linspace(0, 1, 201)[:-1]
dx_m = x_m[1] - x_m[0]
s_g = 0.03
gauss0 = np.exp(-(x_m - 0.5) ** 2 / (2 * s_g**2))
quad0 = np.where(np.abs(x_m - 0.5) < 0.1, 1.0, 0.0)

def quadro_dnum(Cn):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=["gaussiana", "pulso quadrado"])
    D_num = 1.0 * dx_m * (1 - Cn) / 2
    var = s_g**2 + 2 * D_num * 1.0
    previsto = s_g / np.sqrt(var) * np.exp(-(x_m - 0.5) ** 2 / (2 * var))
    for col, c0_ in [(1, gauss0), (2, quad0)]:
        up = adveccao_difusao(c0_, x_m, 1.0, 0.0, 1.0, dt=Cn * dx_m, periodico=True).u[-1]
        lw = adveccao_difusao(c0_, x_m, 1.0, 0.0, 1.0, dt=Cn * dx_m, esquema="lax-wendroff", periodico=True).u[-1]
        mostra = col == 1
        fig.add_trace(go.Scatter(x=x_m, y=c0_, mode="lines", name="exata", showlegend=mostra,
                                 line=dict(color=CORES["exata"], dash="dash")), 1, col)
        fig.add_trace(go.Scatter(x=x_m, y=up, mode="lines", name="upwind", showlegend=mostra,
                                 line=dict(color=CORES["aproximacao"], width=3)), 1, col)
        fig.add_trace(go.Scatter(x=x_m, y=lw, mode="lines", name="Lax-Wendroff", showlegend=mostra,
                                 line=dict(color=CORES["secundaria"], width=2.5)), 1, col)
        if col == 1:
            fig.add_trace(go.Scatter(x=x_m, y=previsto, mode="markers", name="equação modificada (D_num)",
                                     marker=dict(color=CORES["destaque"], size=5)), 1, 1)
    for col in (1, 2):
        fig.update_xaxes(title="x", range=[0.2, 0.8], row=1, col=col)
        fig.update_yaxes(range=[-0.25, 1.25], row=1, col=col)
    fig.update_yaxes(title="c", row=1, col=1)
    fig.update_layout(width=980, height=430, legend=dict(orientation="h", y=-0.22), margin=dict(t=110),
                      title=f"C = {numero(Cn, 2)}: difusão numérica do upwind D_num = {numero(D_num * 1e4, 1)} × 10⁻⁴")
    return fig

quadros = [(numero(v, 2), quadro_dnum(v)) for v in [0.2, 0.5, 0.8, 0.95]]
mostrar(com_controle(quadros, rotulo="número de Courant C = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** os pontos da equação modificada caem sobre a curva do upwind: o upwind resolve, com boa precisão, uma equação de advecção-**difusão**, com uma difusão que nós não pusemos. Ela é menor quanto mais perto $C$ está de 1, o que mostra um fato contraintuitivo: no upwind, **diminuir o passo de tempo piora** a solução. Lax-Wendroff preserva a altura da gaussiana, mas a deixa um pouco atrasada, com uma pequena ondulação atrás; no pulso quadrado, cria oscilações e valores negativos junto às frentes. É a dispersão (as ondas curtas andam mais devagar que as longas) e o teorema de Godunov da seção 3.5.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Simule o derramamento no rio com a equação de advecção-difusão (u = 0,5 m/s, D = 5 m²/s, mancha gaussiana com pico de 50 mg/L e desvio de 150 m no km 2, trecho de 30 km). Calcule a concentração máxima na captação do km 20 e diga se é preciso fechá-la (limite: 10 mg/L)."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import numpy as np

u, D = 0.5, 5.0                  # velocidade (m/s) e dispersão (m²/s)
L, dx = 30000.0, 200.0           # trecho e espaçamento da malha (m)
x = np.arange(0, L + dx, dx)
c = 50.0 * np.exp(-(x - 2000.0) ** 2 / (2 * 150.0**2))   # mancha inicial (mg/L)

dt = 0.5 * dx / u                # passo de tempo (s)
Cr, r = u * dt / dx, D * dt / dx**2
assert Cr + 2 * r <= 1, "passo instável"
print(f"Courant = {Cr:.2f}, r = {r:.3f}: critério de estabilidade C + 2r = {Cr + 2 * r:.2f} <= 1  OK")

t_final = (20000.0 - 2000.0) / u           # tempo de chegada à captação
i_capt = int(round(20000.0 / dx))
c_max = 0.0
for n in range(int(round(t_final / dt * 1.2))):
    # upwind para a advecção (u > 0), centrado para a difusão
    c[1:-1] = c[1:-1] - Cr * (c[1:-1] - c[:-2]) + r * (c[2:] - 2 * c[1:-1] + c[:-2])
    c_max = max(c_max, c[i_capt])

print(f"Concentração máxima na captação: {c_max:.2f} mg/L")
print("Recomendação:", "FECHAR a captação." if c_max > 10 else "manter a captação aberta (abaixo do limite).")

O código está bem escrito: usa o upwind, que é a escolha certa para a advecção (seção 3.4), verifica a condição de estabilidade com a difusão, e o resultado é físico: nada negativo, nada explode. A conclusão é **manter a captação aberta**. A solução exata da seção 1 dá 12,13 mg/L. Se não tivéssemos essa referência, como descobriríamos o erro?

### 6.2 Verificação

**Conservação da massa.** A massa derramada, $\int c\,dx$, tem de se manter. É o primeiro teste que um engenheiro faria:

In [ ]:
massa = lambda conc, malha: np.trapezoid(conc, malha)
x_ag = np.arange(0, L_rio + 200.0, 200.0)
c_ag = adveccao_difusao(c_exata(x_ag, 0), x_ag, u_rio, D_rio, T_chegada, dt=200.0).u[-1]   # o esquema do agente
print(f"massa inicial: {massa(c_exata(x_ag, 0), x_ag):.2f} (mg/L)·m;  massa ao chegar à captação: {massa(c_ag, x_ag):.2f}")
print(f"pico na captação, no instante da chegada: {c_ag[100]:.2f} mg/L")

A massa se conserva até a última casa. O upwind é conservativo (cada célula passa à vizinha exatamente o que perde), e por isso **passa** nesse teste, mesmo errado. Um teste que um código errado passa não verifica nada sobre o erro em questão.

**Refinamento da malha.** O teste que nunca falha em mostrar um problema de discretização, mantendo $C = 0{,}5$:

In [ ]:
print("  Δx (m)   pico upwind (mg/L)")
picos_up = {}
for dx_ in [400, 200, 100, 50, 25]:
    xx = np.arange(0, L_rio + dx_ / 2, dx_)
    res = adveccao_difusao(c_exata(xx, 0), xx, u_rio, D_rio, T_chegada, dt=0.5 * dx_ / u_rio)
    picos_up[dx_] = res.u[-1][np.argmin(np.abs(xx - x_capt))]
    print(f"{dx_:7.0f} {picos_up[dx_]:14.2f}")

O pico **não converge**: a cada refinamento sobe 1 a 1,5 mg/L, e com 25 m (1 200 células, 1 440 passos) ainda está abaixo do limite. Os 5 mg/L do agente são uma propriedade da malha, não do rio.

**O espalhamento.** A teoria diz que a variância da mancha cresce como $\sigma^2 = \sigma_0^2 + 2Dt$. Medindo a variância da solução do agente, descobrimos com que dispersão ela foi calculada:

In [ ]:
m0 = massa(c_ag, x_ag)
centro = massa(c_ag * x_ag, x_ag) / m0
variancia = massa(c_ag * (x_ag - centro) ** 2, x_ag) / m0
D_efetivo = (variancia - sigma0**2) / (2 * T_chegada)
print(f"centro da mancha: km {centro / 1000:.2f} (exato: km 20,00)")
print(f"dispersão efetiva da simulação: {D_efetivo:.2f} m²/s (física: {D_rio} m²/s)")

O centro da mancha está certo, mas a simulação espalhou a mancha como se a dispersão fosse **30 m²/s**, seis vezes a física. Os 25 m²/s a mais são a difusão numérica.

### 6.3 Por quê

A equação modificada {eq}`eq-dnum` prevê exatamente esse valor: $D_{\text{num}} = u\,\Delta x\,(1 - C)/2 = 0{,}5 \cdot 200 \cdot 0{,}5/2 = 25$ m²/s. O upwind é estável, conservativo e monotônico, mas de **primeira ordem**, e seu erro tem a forma de uma difusão. Num problema em que a física é pouco difusiva, o número que controla a qualidade é a razão $D_{\text{num}}/D = \tfrac12\,\text{Pe}_{\Delta x}\,(1 - C)$, em que $\text{Pe}_{\Delta x} = u\,\Delta x/D$ é o **número de Péclet da malha**. Aqui $\text{Pe}_{\Delta x} = 20$: para a difusão numérica ser 1 % da física, o upwind precisaria de $\Delta x \approx 2$ m e cerca de um milhão de passos.

O erro, além disso, é perigoso no sentido em que erra: a difusão numérica **sempre** achata os picos. Numa decisão de segurança, o modelo erra a favor do risco.

### 6.4 A correção

Um esquema de segunda ordem, em que o erro principal não é difusivo. Lax-Wendroff, com o refinamento como verificação:

In [ ]:
print("  Δx (m)   pico LW (mg/L)   erro máximo no perfil (mg/L)   menor concentração (mg/L)")
erros_lw, dxs_lw = [], np.array([400, 200, 100, 50, 25])
for dx_ in dxs_lw:
    xx = np.arange(0, L_rio + dx_ / 2, dx_)
    res = adveccao_difusao(c_exata(xx, 0), xx, u_rio, D_rio, T_chegada, dt=0.5 * dx_ / u_rio, esquema="lax-wendroff")
    p = res.u[-1][np.argmin(np.abs(xx - x_capt))]
    erros_lw.append(np.max(np.abs(res.u[-1] - c_exata(xx, T_chegada))))
    print(f"{dx_:7.0f} {p:14.3f} {erros_lw[-1]:22.4f} {res.u[-1].min():26.3f}")
print(f"ordem observada nas três malhas mais finas: {ordem_observada(dxs_lw[2:], erros_lw[2:]):.2f}")

Lax-Wendroff não é mágico: na malha de 200 m do agente ainda dá 9,4 mg/L (a decisão continua errada) e produz concentrações negativas, a dispersão da seção 3.5. Mas ele converge muito mais depressa: com 100 m o pico erra por 5 % e a decisão já é a correta; com 50 m, por menos de 1 %, e sem valores negativos; cada refinamento divide o erro por 4, a ordem 2. O upwind, para errar o pico por 5 %, precisaria de uma malha de poucos metros (exercício: estime-a pela equação modificada). A recomendação correta: **fechar a captação entre 9,8 h e 10,2 h** após o derramamento, com margem, porque num rio real $u$ e $D$ são incertos.

> **Lição.** Estabilidade e conservação **não** garantem precisão. Para problemas dominados pela advecção, peça ao agente o **número de Péclet da malha** e a difusão numérica $D_{\text{num}}$ do esquema, compare-a com a física, e exija um **estudo de refinamento** da quantidade de interesse (aqui, o pico na captação). Um esquema de primeira ordem achata picos e frentes: num problema de segurança, o erro está sempre do lado perigoso.

## 7. Quando falha

### 7.1 Euler com diferenças centradas na advecção

A escolha "óbvia" para $c_t + u\,c_x = 0$, Euler explícito no tempo e diferença centrada no espaço, é instável para qualquer passo (seção 3.4). O crescimento é lento com $C$ pequeno, o que o torna traiçoeiro: a simulação parece boa por algum tempo.

In [ ]:
for Cn in [0.5, 0.1, 0.02]:
    res = adveccao_difusao(quad0, x_m, 1.0, 0.0, 5.0, dt=Cn * dx_m, esquema="centrado", periodico=True,
                           instantes=np.arange(0.01, 5.0, 0.01))
    amp = np.abs(res.u).max(axis=1)
    t10 = res.t[np.argmax(amp > 10)]
    print(f"C = {Cn:4}: |c| passa de 10 em t = {t10:.2f} (depois de {round(t10 / (Cn * dx_m))} passos); "
          f"em t = 1 (uma volta), máximo de |c| = {amp[np.argmin(np.abs(res.t - 1))]:.1e}")

Com $C = 0{,}02$, o pulso ainda parece razoável depois de uma volta completa, e explode depois. Reduzir o passo só **adia** a explosão: o fator por passo, $\sqrt{1 + C^2\sin^2\theta}$, fica perto de 1, mas o número de passos cresce na mesma proporção. A condição de estabilidade não é sobre o tamanho do passo, e sim sobre a estrutura do esquema.

### 7.2 Diferenças centradas e o número de Péclet da malha

Com difusão, o esquema centrado fica estável se $C^2 \le 2r \le 1$. Mas estável não quer dizer bom: no rio, com a malha do agente,

In [ ]:
for dx_ in [200, 50, 10]:
    xx = np.arange(0, L_rio + dx_ / 2, dx_)
    res = adveccao_difusao(c_exata(xx, 0), xx, u_rio, D_rio, T_chegada, esquema="centrado")
    print(f"Δx = {dx_:3d} m, Pe da malha = {u_rio * dx_ / D_rio:4.0f}: pico na captação {res.u[-1][np.argmin(np.abs(xx - x_capt))]:6.2f} mg/L, "
          f"menor concentração {res.u[-1].min():7.2f} mg/L, {res.passos} passos")

Com $\text{Pe}_{\Delta x} > 2$, o esquema centrado produz concentrações **negativas**, de vários mg/L: as mesmas oscilações da camada-limite do capítulo 14 (seção 7.3). Com $\text{Pe}_{\Delta x} = 1$, as oscilações somem, mas o pico sai 13 % **acima** do exato: o Euler explícito no tempo acrescenta uma difusão **negativa**, $-u^2\Delta t/2$, que é justamente o termo que Lax-Wendroff corrige. Nem a estabilidade nem a ausência de oscilações bastam; a ordem do esquema no tempo também conta.

### 7.3 Crank-Nicolson com dados descontínuos

Uma barra inteira a 1, com as pontas mergulhadas em gelo (0) no instante inicial: o dado inicial é descontínuo nas pontas. Com um passo grande, que BTCS e Crank-Nicolson permitem ($\Delta x = 0{,}01$, $\Delta t = 0{,}01$, ou seja, $r = 100$):

In [ ]:
x_c = np.linspace(0, 1, 101)
u_c0 = np.ones(101)
u_c0[[0, -1]] = 0.0
inst = [0.01, 0.02, 0.03, 0.04]
ref_c = calor_theta(u_c0, x_c, 1.0, 0.04, 1e-5, 1.0, instantes=inst)         # referência com passo pequeno
res_c = {}
for theta, nome in [(0.5, "Crank-Nicolson"), (1.0, "BTCS")]:
    res_c[nome] = res = calor_theta(u_c0, x_c, 1.0, 0.04, 0.01, theta, instantes=inst)
    print(f"{nome:15s} menor temperatura: " + ", ".join(f"t = {t:.2f}: {v.min():+.3f}" for t, v in zip(res.t[1:], res.u[1:])))

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=["Crank-Nicolson, Δt = 0,01", "BTCS, Δt = 0,01"])
cores_c = [CORES["aproximacao"], CORES["secundaria"], CORES["terciaria"], CORES["referencia"]]
for col, nome in [(1, "Crank-Nicolson"), (2, "BTCS")]:
    res = res_c[nome]
    for j, (t, v) in enumerate(zip(res.t[1:], res.u[1:])):
        fig.add_trace(go.Scatter(x=x_c, y=v, mode="lines+markers", marker=dict(size=4), name=f"t = {numero(t, 2)}",
                                 line=dict(color=cores_c[j], width=2), showlegend=(col == 1)), 1, col)
        fig.add_trace(go.Scatter(x=x_c, y=ref_c.u[j + 1], mode="lines", showlegend=False,
                                 line=dict(color=cores_c[j], dash="dot", width=1)), 1, col)
    fig.update_xaxes(title="x", range=[0, 0.3], row=1, col=col)
fig.update_yaxes(title="u", range=[-0.85, 1.05], row=1, col=1)
fig.update_yaxes(range=[-0.85, 1.05], row=1, col=2)
fig.update_layout(width=980, height=420, legend=dict(orientation="h", y=-0.22),
                  title="Perto da ponta fria: Crank-Nicolson oscila (pontilhado: referência com passo pequeno)")
mostrar(fig)

O Crank-Nicolson produz temperaturas **abaixo de 0**, menores que a do gelo, junto às pontas; elas trocam de sinal a cada passo e custam a desaparecer: os modos de alta frequência criados pela descontinuidade têm $g_{\text{CN}} \approx -1$ (seção 3.3). O BTCS, menos preciso em teoria, amortece esses modos e dá uma solução física. As soluções práticas: alguns passos iniciais de BTCS antes de passar ao Crank-Nicolson (a estratégia de Rannacher), ou passos menores no início.

### 7.4 Burgers fora da forma conservativa

Um degrau, $u = 1$ à esquerda e $u = 0$ à direita, na equação de Burgers. A teoria (seção 3.6) diz que é um choque com velocidade $(1 + 0)/2 = 0{,}5$. Comparamos o mesmo esquema upwind de primeira ordem nas duas formas:

In [ ]:
x_bg = np.linspace(-1, 3, 401)
casos_bg = {}
for uD in (0.0, 0.5):
    u0_bg = np.where(x_bg < 0, 1.0, uD)
    s_ex = (1 + uD) / 2                                   # velocidade do choque (Rankine-Hugoniot)
    exata_bg = np.where(x_bg < s_ex * 2, 1.0, uD)
    cons = burgers(u0_bg, x_bg, 2.0, 0.005, "conservativa").u[-1]
    nao = burgers(u0_bg, x_bg, 2.0, 0.005, "nao-conservativa").u[-1]
    casos_bg[uD] = (exata_bg, cons, nao)
    frente = lambda v: x_bg[np.argmax(v < (1 + uD) / 2)]
    print(f"u_D = {uD}: frente em t = 2 na posição exata {2 * s_ex:.2f}; conservativa {frente(cons):.2f}; "
          f"não conservativa {frente(nao):.2f}")

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=["u = 1 → 0: choque com velocidade 0,5", "u = 1 → 0,5: choque com velocidade 0,75"])
for col, uD in [(1, 0.0), (2, 0.5)]:
    exata_bg, cons, nao = casos_bg[uD]
    for y, nome, cor, traco in [(exata_bg, "exata", CORES["exata"], "dash"), (cons, "forma conservativa", CORES["aproximacao"], "solid"),
                                (nao, "forma não conservativa", CORES["erro"], "solid")]:
        fig.add_trace(go.Scatter(x=x_bg, y=y, mode="lines", name=nome, showlegend=(col == 1),
                                 line=dict(color=cor, dash=traco, width=2.5)), 1, col)
    fig.update_xaxes(title="x", row=1, col=col)
fig.update_yaxes(title="u", row=1, col=1)
fig.update_layout(width=980, height=420, title="Equação de Burgers em t = 2: a forma não conservativa erra a posição do choque",
                  legend=dict(orientation="h", y=-0.22))
mostrar(fig)

A forma não conservativa, $u_i^{n+1} = u_i^n - \tfrac{\Delta t}{\Delta x}\,u_i^n(u_i^n - u_{i-1}^n)$, parece uma discretização perfeitamente razoável de $u_t + u\,u_x = 0$. No primeiro caso, o choque **não se move**: em cada célula, ou $u_i = 0$ ou $u_i - u_{i-1} = 0$. No segundo, ele anda, mas devagar demais, e o erro não diminui com o refinamento: o esquema converge para a solução **errada**. Em códigos de escoamento compressível, de tráfego ou de águas rasas, use sempre a forma conservativa (volumes finitos), e peça ao agente que verifique a velocidade do choque num problema de Riemann com solução conhecida.

## Resumo

| Equação | Esquema | Ordem (tempo, espaço) | Estabilidade | Cuidado |
| --- | --- | --- | --- | --- |
| Laplace/Poisson | 5 pontos + direto esparso | —, 2 | — | memória em 3D |
| Laplace/Poisson | Jacobi / Gauss-Seidel / SOR | —, 2 | convergem (matriz diagonal dominante) | iterações $\propto n^2$ (SOR: $n$); critério de parada |
| calor | FTCS | 1, 2 | $r \le 1/2$ | $\Delta t \propto \Delta x^2$ |
| calor | BTCS | 1, 2 | sempre | sistema tridiagonal por passo |
| calor | Crank-Nicolson | 2, 2 | sempre (A-estável) | oscilações com dados descontínuos |
| advecção | centrado (FTCS) | — | **nunca** | não use |
| advecção(-difusão) | upwind | 1, 1 | $C + 2r \le 1$ | difusão numérica $u\Delta x(1-C)/2$ |
| advecção(-difusão) | Lax-Wendroff | 2, 2 | $C^2 + 2r \le 1$ | oscilações em frentes (dispersão) |
| leis de conservação | volumes finitos (Godunov) | 1, 1 | $\max\lvert u\rvert\,\Delta t/\Delta x \le 1$ | forma conservativa obrigatória |

## Exercícios

**Conceituais (sem computador)**

1. Classifique $u_{xx} + 4u_{xy} + u_{yy} = 0$ e $y\,u_{xx} + u_{yy} = 0$ (esta, em cada região do plano).
2. Faça a análise de von Neumann do BTCS e do Crank-Nicolson e confira as expressões de $g$ da seção 3.3. Para que valor de $r$ o Crank-Nicolson amortece o modo $\theta = \pi$ por um fator 0,5 a cada passo?
3. Deduza a equação modificada do upwind, {eq}`eq-dnum`. Mostre que, com $C = 1$, o erro de truncamento é zero.
4. Para a advecção-difusão com upwind, mostre que a condição de estabilidade $C + 2r \le 1$ também garante que $c_i^{n+1}$ é uma média ponderada (com pesos não negativos) de $c_{i-1}^n$, $c_i^n$ e $c_{i+1}^n$, e conclua que o esquema não cria máximos nem mínimos novos.

**Verificação (com agente)**

5. Peça a um agente a distribuição de temperatura numa placa em L (por exemplo, a seção transversal de uma chaminé) com o estêncil de 5 pontos. Verifique pelo princípio do máximo, pelo refinamento da malha e pela ordem observada; o que acontece com a ordem perto do canto reentrante?
6. Peça a um agente para resolver o rio com um esquema de alta resolução (com limitador, por exemplo *minmod* ou *superbee*). Compare o pico na captação e os valores mínimos com o upwind e com Lax-Wendroff, nas malhas de 400 e 200 m.
7. Peça a um agente para simular o tráfego numa estrada com o modelo LWR, $\rho_t + [\rho\,v_{\max}(1 - \rho/\rho_{\max})]_x = 0$, num semáforo que abre (rarefação) e num engarrafamento que se forma (choque). Verifique a velocidade do choque pela condição de Rankine-Hugoniot.

**Projeto aberto**

8. **Plano de emergência.** Estenda o modelo do rio para velocidade e dispersão variáveis ao longo do trecho (por exemplo, $u$ menor num remanso entre os km 8 e 12) e para um derramamento contínuo de uma hora em vez de instantâneo. Escolha e verifique um esquema, e produza, para a companhia de saneamento, uma tabela de "horário de fechamento" da captação em função da hora e da massa do derramamento, com uma margem de segurança justificada.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 29 e 30.
- R. J. LeVeque, *Finite Difference Methods for Ordinary and Partial Differential Equations*, SIAM, 2007, capítulos 9 e 10.
- R. J. LeVeque, *Finite Volume Methods for Hyperbolic Problems*, Cambridge University Press, 2002 (leis de conservação, Godunov, limitadores).
- H. B. Fischer e outros, *Mixing in Inland and Coastal Waters*, Academic Press, 1979 (dispersão longitudinal em rios).